<a href="https://colab.research.google.com/github/Esra-BR/AIM_460/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import re
import json
from pathlib import Path

RAW_REG_DIR  = Path("suffolk_data/raw/regulations")
OUT_REG_DIR  = Path("suffolk_data/regulations")
OUT_REG_DIR.mkdir(parents=True, exist_ok=True)

# --- KEYWORDS ---
OBLIGATION_WORDS = ["shall", "must", "required", "requires", "prohibited"]
TEMP_KEYWORDS = ["temperature", "hot", "cold", "refrigeration"]
PERMIT_KEYWORDS = ["permit", "license", "certificate"]
PENALTY_KEYWORDS = ["fine", "penalty", "closure", "violation"]
MOBILE_KEYWORDS = ["mobile", "food truck", "cart", "vendor"]
ALLERGEN_KEYWORDS = ["milk", "dairy", "peanut", "egg"]

# --- BUILT-IN RULES ---
BUILTIN_RULES = [
    {
        "rule_id": "MV-001",
        "topic": "permit",
        "obligation": "Mobile vendors must obtain a permit before operating.",
        "condition": "Permit must be displayed.",
        "penalty": "Operation without permit leads to closure.",
        "mobile_specific": True
    },
    {
        "rule_id": "MV-002",
        "topic": "temperature_control",
        "obligation": "Hot food must be kept above 140F.",
        "condition": "Applies during service.",
        "penalty": "Critical violation.",
        "mobile_specific": True
    },
    {
        "rule_id": "MV-003",
        "topic": "temperature_control",
        "obligation": "Cold food must be kept below 45F.",
        "condition": "Requires refrigeration.",
        "penalty": "Critical violation.",
        "mobile_specific": True
    },
    {
        "rule_id": "MV-004",
        "topic": "sanitation",
        "obligation": "Handwashing stations are required.",
        "condition": "Must include soap and water.",
        "penalty": "Critical violation.",
        "mobile_specific": True
    },
    {
        "rule_id": "MV-005",
        "topic": "cross_contamination",
        "obligation": "Raw meat must be separated from ready-to-eat foods.",
        "condition": "Use separate storage.",
        "penalty": "Critical violation.",
        "mobile_specific": True
    }
]

# --- FOOD RULE MAPPING (THIS IS IMPORTANT FOR YOUR ROLE) ---
FOOD_RULE_MAP = {
    "meat": ["temperature_control", "cross_contamination"],
    "dairy": ["temperature_control", "refrigeration", "allergen_labeling"],
    "milk": ["refrigeration", "allergen_labeling"],
    "cheese": ["refrigeration"],
    "propane": ["permit", "fire_safety"],
    "grill": ["fire_safety"]
}

# --- PERMIT CHECKLIST ---
PERMIT_CHECKLIST = [
    {"step": 1, "item": "Permit application", "required": True},
    {"step": 2, "item": "Food handler certificate", "required": True},
    {"step": 3, "item": "Inspection approval", "required": True}
]

# --- EXTRACTION FUNCTION ---
def extract_from_text(text):
    rules = []
    sentences = re.split(r'(?<=[.!?])\s+', text)

    for i, sent in enumerate(sentences):
        sent_lower = sent.lower()

        if not any(word in sent_lower for word in OBLIGATION_WORDS):
            continue

        rule = {
            "rule_id": f"EXT-{i}",
            "raw_text": sent,
            "mobile_specific": any(w in sent_lower for w in MOBILE_KEYWORDS),
            "topics": []
        }

        if any(w in sent_lower for w in TEMP_KEYWORDS):
            rule["topics"].append("temperature_control")
        if any(w in sent_lower for w in PERMIT_KEYWORDS):
            rule["topics"].append("permit")
        if any(w in sent_lower for w in PENALTY_KEYWORDS):
            rule["topics"].append("penalty")
        if any(w in sent_lower for w in ALLERGEN_KEYWORDS):
            rule["topics"].append("allergen_labeling")

        # Add structured fields (IMPORTANT FOR YOUR ROLE)
        rule["obligation"] = sent
        rule["condition"] = None
        rule["penalty"] = sent if "penalty" in rule["topics"] else None

        rules.append(rule)

    return rules

# --- LOAD TEXT FILES IF THEY EXIST ---
def extract_all():
    all_rules = []
    txt_files = list(RAW_REG_DIR.glob("*.txt"))

    if txt_files:
        for file in txt_files:
            text = file.read_text()
            all_rules.extend(extract_from_text(text))

    # Always include built-in rules
    all_rules.extend(BUILTIN_RULES)
    return all_rules

# --- SAVE OUTPUTS ---
def run():
    rules = extract_all()

    mobile_rules = [r for r in rules if r.get("mobile_specific")]

    (OUT_REG_DIR / "extracted_rules.json").write_text(json.dumps(rules, indent=2))
    (OUT_REG_DIR / "mobile_vendor_rules.json").write_text(json.dumps(mobile_rules, indent=2))
    (OUT_REG_DIR / "permit_checklist.json").write_text(json.dumps(PERMIT_CHECKLIST, indent=2))
    (OUT_REG_DIR / "food_rule_map.json").write_text(json.dumps(FOOD_RULE_MAP, indent=2))

    print("Done. Your NLP module is working.")

# --- RUN ---
if __name__ == "__main__":
    run()

Done. Your NLP module is working.


In [7]:
from google.colab import files
files.download("suffolk_data/regulations/extracted_rules.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
files.download("suffolk_data/regulations/mobile_vendor_rules.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
files.download("suffolk_data/regulations/food_rule_map.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
from google.colab import files
files.download("suffolk_data/regulations/permit_checklist.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>